## Analyze the impact of optimization, weight Initialization and regularization


### 1. Experimenting with Optimization Algorithms on Fashion-MNIST

> **CHANGE:** This experiment uses Fashion-MNIST instead of CIFAR-10; the optimizer/initializer/regularization comparisons are otherwise preserved.

##### Import Necessary Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist  # CHANGE: Fashion-MNIST dataset
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers  import Dense, Flatten
from tensorflow.keras.optimizers import  SGD, RMSprop, Adagrad, Adam
import matplotlib.pyplot as plt



#### plot learning curves- function

In [2]:
import matplotlib.pyplot as plt

def plot_training_history(history, header):
    """
    Plots training and validation accuracy and loss across epochs, with a header based on the options.

    Args:
        history: Training history object returned by the model.fit() method.
        header: String specifying the option.
    """
    plt.figure(figsize=(12, 4))

    # Plot accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'Accuracy {header}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'Loss {header}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.show()


##### Load and Preprocess Data

In [3]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()  # CHANGE: load Fashion-MNIST

# Normalize pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

# Convert class vectors to binary class matrices

num_classes = 10
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


##### Define the Model

In [4]:
def create_model():
    model = Sequential([
      # CHANGE: Fashion-MNIST images are 28x28 grayscale
      Flatten(input_shape=(28, 28)),
      Dense(512, activation='relu'),
      Dense(256, activation='relu'),
      Dense(128, activation='relu'),
      Dense(num_classes, activation='softmax')
    ])
    return model

##### Define Optimization Algorithms and Hyperparameters

In [5]:
optimizers = {
    'GD': SGD(learning_rate=0.01),
    'SGD': SGD(learning_rate=0.01, momentum=0.9),
    'MiniBatch': SGD(learning_rate=0.01),  # Mini-batch is handled by batch_size parameter
    'RMSprop': RMSprop(learning_rate=0.001),
    'Adagrad': Adagrad(learning_rate=0.01),
    'Adam': Adam(learning_rate=0.001)
}

epochs = 10
batch_size = 32

##### Training and Evaluation Loop

In [6]:
def train_and_evaluate(optimizer):
  model = create_model()
  model.compile(loss='categorical_crossentropy',
                optimizer=optimizer,
                metrics=['accuracy'])

  history = model.fit(x_train, y_train,
                    batch_size=batch_size,
                    epochs=epochs,
                    validation_data=(x_test, y_test))

  # Plot learning curves here using your plot_training_history function
  # plot_training_history(history,'(optimizer: {optimizer}' )

  test_loss, test_acc = model.evaluate(x_test, y_test)
  print('Test accuracy:', test_acc)
  return test_acc, history

##### Iterate Over Optimizers

In [7]:
# @title
results = {}
for optimizer_name, optimizer in optimizers.items():
  print(f"Training with optimizer: {optimizer_name}")
  accuracy, history = train_and_evaluate(optimizer)
  results[optimizer_name] = accuracy


Training with optimizer: GD


/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.7744 - loss: 0.6842 - val_accuracy: 0.8058 - val_loss: 0.5463
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8395 - loss: 0.4607 - val_accuracy: 0.8386 - val_loss: 0.4555
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8544 - loss: 0.4150 - val_accuracy: 0.8475 - val_loss: 0.4313
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8637 - loss: 0.3852 - val_accuracy: 0.8515 - val_loss: 0.4159
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8713 - loss: 0.3635 - val_accuracy: 0.8592 - val_loss: 0.3901
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8754 - loss: 0.3463 - val_accuracy: 0.8546 - val_loss: 0.3982
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8806 - loss: 0.3303 - val_accuracy: 0.8689 - val_loss: 0.3670
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8850 - loss: 0.3191 - 

##### Compare results

In [8]:
print(results)

{'GD': 0.8726999759674072, 'SGD': 0.8851000070571899, 'MiniBatch': 0.871999979019165, 'RMSprop': 0.8646000027656555, 'Adagrad': 0.8851000070571899, 'Adam': 0.8847000002861023}


## 2. Experimenting with Initialization Techniques on Fashion-MNIST
This code explores the impact of different initialization techniques (Xavier and He normalization) on a feedforward neural network for Fashion-MNIST classification in Keras.

> **CHANGE:** This experiment uses Fashion-MNIST instead of CIFAR-10; the optimizer/initializer/regularization comparisons are otherwise preserved.

##### Imports

In [9]:
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist  # CHANGE: Fashion-MNIST dataset
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.initializers  import glorot_uniform, he_normal

import matplotlib.pyplot as plt

##### Load and Preprocess Data

In [10]:

# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()  # CHANGE: load Fashion-MNIST

# Normalize pixel values to    range [0, 1]
x_train, x_test = x_train / 255.0, x_test / 255.0

# Convert class vectors to binary class matrices
num_classes = 10
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

##### Define the Model

In [11]:
def create_model(init_type):
  model = Sequential([
    # CHANGE: Fashion-MNIST images are 28x28 grayscale
      Flatten(input_shape=(28, 28)),
    Dense(512, activation='relu', kernel_initializer=init_type),
    Dense(256, activation='relu', kernel_initializer=init_type),
    Dense(128, activation='relu', kernel_initializer=init_type),
    Dense(num_classes, activation='softmax')
  ])
  return model

##### Define Training and Evaluation Function

In [12]:
def train_and_evaluate(init_type):
  model = create_model(init_type)
  model.compile(loss='categorical_crossentropy',
                optimizer='adam',
                metrics=['accuracy'])

  history = model.fit(x_train, y_train,
                    epochs=10,
                    batch_size=32,
                    validation_data=(x_test, y_test))


  # plot_training_history(history, '(Initialization: {init_type}')  # Call your learning curve plotting function

  test_loss, test_acc = model.evaluate(x_test, y_test)
  print('Test accuracy:', test_acc)

  return test_acc

##### Experiment with Initializers

In [13]:
initializers = {
  'Xavier': glorot_uniform,
  'HeNormal': he_normal
}

results = {}
for init_name, init_func in initializers.items():
  print(f"Training with initialization: {init_name}")
  accuracy = train_and_evaluate(init_func)
  results[init_name] = accuracy

Training with initialization: Xavier
Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.8275 - loss: 0.4772 - val_accuracy: 0.8523 - val_loss: 0.4009
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8673 - loss: 0.3624 - val_accuracy: 0.8646 - val_loss: 0.3730
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8802 - loss: 0.3244 - val_accuracy: 0.8763 - val_loss: 0.3511
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8874 - loss: 0.3014 - val_accuracy: 0.8661 - val_loss: 0.3595
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8942 - loss: 0.2836 - val_accuracy: 0.8772 - val_loss: 0.3475
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8996 - loss: 0.2697 - val_accuracy: 0.8850 - val_loss: 0.3369
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9032 - loss: 0.2590 - val_accuracy: 0.8792 - val_loss: 0.3431
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step

##### Compare Results

In [14]:
print(results)

{'Xavier': 0.8794000148773193, 'HeNormal': 0.883400022983551}


## 3. Experimenting with Regularization Techniques on Fashion-MNIST
This code explores the effects of different regularization techniques (L2 weight decay, dropout, and early stopping) on a feedforward neural network for Fashion-MNIST image classification in Keras.

> **CHANGE:** This experiment uses Fashion-MNIST instead of CIFAR-10; the optimizer/initializer/regularization comparisons are otherwise preserved.

##### Imports

In [15]:
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist  # CHANGE: Fashion-MNIST dataset
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers  import Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt

##### Load and Preprocess Data

In [16]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()  # CHANGE: load Fashion-MNIST

# Normalize pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

# Convert class vectors to binary class matrices

num_classes = 10
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

##### Define the Model

In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten

def create_model(regularization_type, l2_reg=0.0):
    model = Sequential()
    # CHANGE: Fashion-MNIST images are 28x28 grayscale
    model.add(Flatten(input_shape=(28, 28)))

    # First Dense layer
    model.add(Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2_reg) if regularization_type == 'L2' else None))
    if regularization_type == 'Dropout':
        model.add(Dropout(0.2))

    # Second Dense layer
    model.add(Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2_reg) if regularization_type == 'L2' else None))
    if regularization_type == 'Dropout':
        model.add(Dropout(0.2))

    # Third Dense layer
    model.add(Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2_reg) if regularization_type == 'L2' else None))

    # Output layer
    model.add(Dense(num_classes, activation='softmax'))

    return model


##### Define Training and Evaluation Function

In [18]:
def train_and_evaluate(regularization_type, l2_reg=0.0, patience=5):
  model = create_model(regularization_type, l2_reg)
  model.compile(loss='categorical_crossentropy',
                optimizer='adam',
                metrics=['accuracy'])

  early_stopping = EarlyStopping(monitor='val_accuracy', patience=patience)  # Early stopping callback

  history = model.fit(x_train, y_train,
                    epochs=20,  # May need adjustments based on stopping criteria
                    batch_size=32,
                    validation_data=(x_test, y_test),
                    callbacks=[early_stopping])

  # plot_training_history(history, regularization_type)  # Call your learning curve plotting function

  test_loss, test_acc = model.evaluate(x_test, y_test)
  print('Test accuracy:', test_acc)

  return test_acc

##### Experiment with Regularization Techniques

In [19]:
regularization_options = {
    'Baseline': None,  # No regularization
    'L2': 0.001,
    'Dropout': None,  # Dropout is applied in the model creation function
    'EarlyStopping': None  # Patience defined in the training function
}

results = {}
for reg_name, reg_param in regularization_options.items():
    print(f"Training with regularization: {reg_name}")
    if reg_name == 'EarlyStopping':
        accuracy = train_and_evaluate(reg_name, patience=10)  # Experiment with patience value
    else:
        accuracy = train_and_evaluate(reg_name, reg_param)
    results[reg_name] = accuracy


Training with regularization: Baseline
Epoch 1/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.8284 - loss: 0.4749 - val_accuracy: 0.8373 - val_loss: 0.4430
Epoch 2/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8666 - loss: 0.3611 - val_accuracy: 0.8580 - val_loss: 0.3957
Epoch 3/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8795 - loss: 0.3256 - val_accuracy: 0.8630 - val_loss: 0.3695
Epoch 4/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8875 - loss: 0.3006 - val_accuracy: 0.8666 - val_loss: 0.3742
Epoch 5/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8938 - loss: 0.2849 - val_accuracy: 0.8760 - val_loss: 0.3501
Epoch 6/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8981 - loss: 0.2692 - val_accuracy: 0.8692 - val_loss: 0.3704
Epoch 7/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9020 - loss: 0.2582 - val_accuracy: 0.8796 - val_loss: 0.3399
Epoch 8/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/st

##### Compare Results

In [20]:
print(results)

{'Baseline': 0.8848000168800354, 'L2': 0.8644999861717224, 'Dropout': 0.8883000016212463, 'EarlyStopping': 0.890500009059906}
